In [74]:
import pandas as pd

In [93]:
file_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Clostridium_difficile.xlsx"
sheet = "test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary'
]
df = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols
)

In [62]:
#df=pd.read_excel( file_path:"E:\PaperLLM\LLMzCor.github.io\DBs\Clostridium_difficile.xlsx", sheet_name: "2012-2022")

In [94]:
df.head()

,PMID,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
0,36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,No,Yes,No,Using antibodies (VHHs) AH3 and AA6 are two po...
1,36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,No,Yes,No,"Administration of the PPAR-γ agonist, pioglita..."
2,36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,No,Yes,No,Use Inulin or pectin as a dietary-based therap...
3,36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,No,Yes,No,The paper studied a protein named PtsHN10M tha...
4,35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,Yes,No,No,This study focused on analyzing Clostridioides...


In [78]:
#df['Alerta'].unique()

In [95]:
df.shape

(50, 7)

In [123]:
df.iloc[0]

PMID                                                                          36466927
Title                                Neutralizing epitopes on Clostridioides diffic...
Abstract                             Toxin A (TcdA) and toxin B (TcdB) are two key ...
1) Antimicrobial Resistance stain                                                   No
2) New treatment                                                                   Yes
3) Immunization                                                                     No
Human_summary                        Using antibodies (VHHs) AH3 and AA6 are two po...
Name: 0, dtype: object

In [124]:
from LLM import Clasificador

In [125]:
clasificador=Clasificador()

In [126]:
clasificador.clasificacion(paper)

HfHubHTTPError: 402 Client Error: Payment Required for url: https://api-inference.huggingface.co/models/mistralai/Mixtral-8x7B-Instruct-v0.1 (Request ID: Root=1-681abeff-6ba4dc9e3da4daad794e8706;c248af6b-0ea9-47a3-a91f-73d9e9fbe5b9)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [ ]:
#Prueba 

paper=df['Abstract'][7]
clasificador.clasificacion(paper)

def append_answer_to_csv(response: str, csv_path: str):
    """
    Añade directamente la respuesta del clasificador al CSV usando pandas.
    Solo requiere pandas y la función builtin open(), sin re ni os.
    """
    # Crear DataFrame con la respuesta cruda
    df = pd.DataFrame({'Answer': [response]})
 # Abrir el archivo en modo append; si está vacío, escribe la cabecera
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        # f.tell() == 0 indica archivo vacío (sin bytes previos)
        df.to_csv(f, header=(f.tell() == 0), index=False)
# Ejemplo de uso:
if __name__ == '__main__':
    # Supongamos que 'paper' ya está definido y clasificador disponible
    # response es la cadena tal como la devuelve el clasificador
    response = clasificador.clasificacion(paper)

    # Añadir la respuesta al CSV (se creará si no existe)
    append_answer_to_csv(response, 'results.csv')
    print("Respuesta añadida a results.csv")

In [121]:
##prueba agregar PMID

def append_answer_to_csv(pmid: int, response: str, csv_path: str):
    """
    Añade directamente el PMID y la respuesta del clasificador al CSV usando pandas.
    """
    # Crear DataFrame con PMID y respuesta
    df_out = pd.DataFrame({
        'PMID':  [pmid],
        'Answer':[response]
    })
    # Abrir el archivo en modo append; si está vacío, escribe la cabecera
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        df_out.to_csv(f, header=(f.tell() == 0), index=False)

if __name__ == '__main__':
    # 1) Extrae el paper y su PMID
    idx = 7
    pmid  = df['PMID'].iloc[idx]
    paper = df['Abstract'].iloc[idx]
   
    # 2) Obtén la respuesta del clasificador
    response = clasificador.clasificacion(paper)

    # 3) Añade al CSV tanto el PMID como la respuesta
    append_answer_to_csv(pmid, response, 'results.csv')
    print(f"PMID {pmid} y su respuesta añadidos a results.csv")

PMID 35939437 y su respuesta añadidos a results.csv


In [ ]:
# 
# def append_answer_to_csv(response: str, csv_path: str):
#     """
#     Añade directamente la respuesta del clasificador al CSV usando pandas.
#     Solo requiere pandas y la función builtin open(), sin re ni os.
#     """
#     # Crear DataFrame con la respuesta cruda
#     df = pd.DataFrame({'Answer': [response]})
#  # Abrir el archivo en modo append; si está vacío, escribe la cabecera
#     with open(csv_path, 'a', newline='', encoding='utf-8') as f:
#         # f.tell() == 0 indica archivo vacío (sin bytes previos)
#         df.to_csv(f, header=(f.tell() == 0), index=False)
# # Ejemplo de uso:
# if __name__ == '__main__':
#     # Supongamos que 'paper' ya está definido y clasificador disponible
#     # response es la cadena tal como la devuelve el clasificador
#     response = clasificador.clasificacion(paper)

#     # Añadir la respuesta al CSV (se creará si no existe)
#     append_answer_to_csv(response, 'results.csv')
#     print("Respuesta añadida a results.csv")

In [122]:
##Prueba Loop

# --- Bucle en chunks de 10 papers ---
csv_path = 'results.csv'
for start in range(0, len(df), 10):
    batch = df.iloc[start:start+10]
    print(f"Procesando abstracts {start+1} a {start+len(batch)}…")
    for _, row in batch.iterrows():
        pmid     = row['PMID']
        abstract = row['Abstract']
        # Llamas al clasificador para cada abstract
        response = clasificador.clasificacion(abstract)
        # Guardas PMID + respuesta
        append_answer_to_csv(pmid, response, csv_path)

print("¡Terminado! Todas las respuestas están en", csv_path)

Procesando abstracts 1 a 10…
Procesando abstracts 11 a 20…
Procesando abstracts 21 a 30…
Procesando abstracts 31 a 40…
Procesando abstracts 41 a 50…
¡Terminado! Todas las respuestas están en results.csv
